# Djuren et al. baseline

This notebook runs our implementation (`djuren.py`, in this folder) of the quadratic polynomial setting from

> Tobias Djuren et al. (2025), *Interpolating splines over triangulated surfaces by blending vertex-centric local geometries*.

We use it as a baseline for comparison with BCS. It does not use the BCS framework: each vertex of the coarse mesh gets a quadratic polynomial (no constant term) fitted by least squares to its one-ring neighbours, and these are blended over each triangle with Djuren et al.'s blend weights and one-ring coordinates.

As explained in our paper Blended Chart Surfaces, the two methods share similarities but theirs is interpolating whilst ours is not necessarily interpolating, and theirs aims for a good visual appearance from a coarse mesh without optimisation, whereas our BCS uses optimisation (Adam) with the aim of making the final surface fit closely to a given IMPLICIT surface.

**How to run:** set `surface_config` in the third code cell to the coarse mesh you want (from `data/surfaces/`) and run the cells in order. The cell that builds the surface uses the paper's quadratic setting (`flag='djuren'`, `function_flag='polynomial'`, `onering_coords_flag='djuren'`, degree 2). It shows the result and saves it to `data/surfaces/<mesh>-djuren.obj`.

The commented-out cells further down try other blend functions and one-ring coordinates. Those are our own variations, not part of Djuren et al.'s method.


In [ ]:
# Run from the repository root, so that `import bcs` and the relative
# data/, configs/ and sdf_weights/ paths work wherever this notebook lives.
import os, sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'bcs').is_dir())
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
sys.path.insert(0, 'notebooks/djuren')  # the standalone Djuren baseline lives next to this notebook
from djuren import Djuren_surface
import numpy as np
import torch
import open3d as o3d

In [ ]:
#surface_config = {'coarse_patches_id' : 'big-cube'}
#surface_config = {'coarse_patches_id' : 'igea500', 'degree' : 2}
surface_config = {'coarse_patches_id' : 'bob500', 'degree' : 2}
#surface_config = {'coarse_patches_id' : 'twisted_torus500', 'degree' : 2}
#surface_config = {'coarse_patches_id' : 'fertility500', 'degree' : 2}

degree = surface_config['degree']
name = surface_config['coarse_patches_id']

my_Djuren_surface = Djuren_surface(surface_config = surface_config, mesh_res=6)



my_Djuren_surface.compute_blend_weights(flag='djuren') #blend type

my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='djuren') #vertex func type, onering coords type
mesh_o3d_coloured, mesh_o3d_plain, unblended_patches = my_Djuren_surface.build_output_surface(colour_flag='angles')
o3d.visualization.draw_geometries([mesh_o3d_plain])

o3d.io.write_triangle_mesh('data/surfaces/'+name+'-djuren.obj', mesh_o3d_plain)












In [ ]:
import trimesh
import numpy as np
import scipy.sparse as sparse
import igl
import matplotlib.pyplot as plt
from plyfile import PlyData, PlyElement

# --- PLY export with custom per-vertex colour attributes ---
def write_custom_colour_ply_file(tm=None, colouringdict=None, filepath=None):
    print('Writing custom PLY to:', filepath)
    tm.export(filepath)
    p = PlyData.read(filepath)
    v = p.elements[0]
    f = p.elements[1]
    
    extra_data_specs = [(name, 'f8', (3,)) for name in colouringdict.keys()]
    a = np.empty(len(v.data), v.data.dtype.descr + extra_data_specs)
    
    for name in v.data.dtype.fields:
        a[name] = v[name]
    
    for colouringname, colouring in colouringdict.items():    
        a[colouringname] = colouring[:, :3]
    
    v = PlyElement.describe(a, 'vertex')
    p = PlyData([v, f], text=True)
    p.write(filepath)


def export_curvature_exact_custom(obj_file_path, output_ply_path, mapping_func, cmap):
    # 1. Load the OBJ
    print(f"Loading mesh: {obj_file_path}")
    tm = trimesh.load_mesh(obj_file_path)

    # 2. Compute EXACT Laplace-Beltrami operators using libigl
    print("Computing exact Cotangent Laplacian and Mass matrices...")
    L = igl.cotmatrix(tm.vertices, tm.faces)
    M = igl.massmatrix(tm.vertices, tm.faces, igl.MASSMATRIX_TYPE_BARYCENTRIC)

    # 3. Manually invert the Mass Matrix (Bypassing the igl.invert_diag error)
    m_diag = M.diagonal()
    m_inv = np.zeros_like(m_diag)
    
    # Safely invert only non-zero areas to prevent divide-by-zero errors
    valid_mask = m_diag > 1e-12
    m_inv[valid_mask] = 1.0 / m_diag[valid_mask]
    
    # Rebuild as a SciPy sparse diagonal matrix
    Minv = sparse.diags(m_inv)

    # 4. Calculate exact displacement (Delta x)
    # The true Laplace-Beltrami operator is Minv * L
    print("Applying operator to vertices...")
    delta_x = Minv.dot(L.dot(tm.vertices))

    # 5. Extract scalar Mean Curvature
    # IMPORTANT SIGN NOTE: libigl's cotmatrix convention makes L point INWARD. 
    # Depending on the colormap orientation, the sign may need flipping to match
    # the convention of bcs.differential.DifferentialModule.
    mean_curvature = -0.5 * np.sum(delta_x * tm.vertex_normals, axis=1)

    # 6. Map to Colors (Float array for 'f8' export)
    print("Mapping colors...")
    flat_colors = cmap(mapping_func(mean_curvature))[:, :3].astype(np.float32) 

    # 7. Standard Visuals
    
    tm.visual.vertex_colors = (flat_colors * 255).astype(np.uint8)

    # 8. Export
    name_key = 'meancurv_colours'
    write_custom_colour_ply_file(
        tm=tm,
        colouringdict={name_key: flat_colors},
        filepath=str(output_ply_path)
    )
    
    tm.export(obj_file_path)
    print("Done!")


# --- Execution ---
if __name__ == "__main__":
    from bcs.colourmappings import mapping12
    
    # Configuration
    input_obj = f'data/surfaces/{name}-djuren.obj'
    output_ply = f'data/surfaces/{name}-djuren.ply'
    
    meancurv_cmap = plt.get_cmap('seismic')
    
    export_curvature_exact_custom(
        input_obj, 
        output_ply, 
        mapping_func=mapping12, 
        cmap=meancurv_cmap
    )

In [ ]:







'''
my_Djuren_surface.compute_blend_weights(flag='djuren') #blend type

my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='djuren') #vertex func type, onering coords type
#mesh_o3d, mesh_o3d_plain = my_Djuren_surface.build_output_surface(colour_flag='blend')
#o3d.visualization.draw_geometries([mesh_o3d])
mesh_o3d, mesh_o3d_plain = my_Djuren_surface.build_output_surface(colour_flag='angles')
o3d.visualization.draw_geometries([mesh_o3d])
'''



'''
my_Djuren_surface.compute_blend_weights(flag='bary') #blend type

my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='djuren') #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='angles')
o3d.visualization.draw_geometries(unblended_o3d)
#o3d.visualization.draw_geometries([mesh_o3d])
o3d.visualization.draw_geometries([mesh_o3d_plain], window_name="bary blend, djuren polynomials")
'''

'''

my_Djuren_surface.compute_blend_weights(flag='bary') #blend type

my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='equal') #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='angles')
o3d.visualization.draw_geometries(unblended_o3d)
#o3d.visualization.draw_geometries([mesh_o3d])
o3d.visualization.draw_geometries([mesh_o3d_plain], window_name="bary blend, equal-angle polynomials")
'''




'''
my_Djuren_surface.compute_blend_weights(flag='djuren') #blend type
my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='djuren', degree=degree) #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='blend')
o3d.visualization.draw_geometries([mesh_o3d_plain],  window_name="djuren blend, djuren polynomials")
#o3d.visualization.draw_geometries([mesh_o3d],  window_name="djuren blend, djuren polynomials")
'''

'''
my_Djuren_surface.compute_blend_weights(flag='inv-exp', overlap_param = 0.99) #blend type
my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='equal', degree=degree) #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='blend')
o3d.visualization.draw_geometries([mesh_o3d_plain],  window_name="inv exp blend, equal-angle polynomials")
o3d.visualization.draw_geometries([mesh_o3d],  window_name="inv exp blend, inv exp polynomials")



my_Djuren_surface.compute_blend_weights(flag='inv-exp') #blend type
my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='djuren', degree=degree) #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='blend')
o3d.visualization.draw_geometries([mesh_o3d_plain],  window_name="inv-exp blend, djuren polynomials")
#o3d.visualization.draw_geometries([mesh_o3d],  window_name="djuren blend, djuren polynomials")

my_Djuren_surface.compute_blend_weights(flag='djuren') #blend type
my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='equal', degree=degree) #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='blend')
o3d.visualization.draw_geometries([mesh_o3d_plain],  window_name="djuren blend, equal-angle polynomials")
#o3d.visualization.draw_geometries([mesh_o3d],  window_name="inv exp blend, equal-angle polynomials")

my_Djuren_surface.compute_blend_weights(flag='djuren') #blend type
my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='equal', degree=degree) #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='angles')
o3d.visualization.draw_geometries([mesh_o3d_plain], window_name="djuren blend, equal-angle polynomials")

my_Djuren_surface.compute_blend_weights(flag='bary') #blend type
my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='djuren', degree=degree) #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='angles')
o3d.visualization.draw_geometries([mesh_o3d_plain], window_name="bary blend, djuren polynomials")

my_Djuren_surface.compute_blend_weights(flag='bary') #blend type
my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='equal', degree=degree) #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='angles')
o3d.visualization.draw_geometries([mesh_o3d_plain], window_name="bary blend, equal-angle polynomials")
'''

#my_Djuren_surface.he_mesh.paint_uniform_color([0, 0, 1])
#for i in range(3*my_Djuren_surface.F.shape[0]):
#    o3d.visualization.draw_geometries([unblended_o3d[i]] + [my_Djuren_surface.he_mesh])

#o3d.visualization.draw_geometries([mesh_o3d])
#o3d.visualization.draw_geometries(unblended_o3d + [my_Djuren_surface.he_mesh])
#o3d.visualization.draw_geometries([mesh_o3d])
#my_Djuren_surface.he_mesh.paint_uniform_color([0, 0, 1])


'''
my_Djuren_surface.compute_blend_weights(flag='djuren') #blend type

my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='djuren') #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='angles')
o3d.visualization.draw_geometries(unblended_o3d)
o3d.visualization.draw_geometries([mesh_o3d])
o3d.visualization.draw_geometries([mesh_o3d_plain])
'''


'''
##### Ours: smooth.
my_Djuren_surface.compute_blend_weights(flag='inv-exp') #blend type

my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='equal') #vertex func type, onering coords type
mesh_o3d_angles, mesh_o3d_plain = my_Djuren_surface.build_output_surface()

o3d.visualization.draw_geometries([mesh_o3d_angles])
o3d.visualization.draw_geometries([mesh_o3d_plain])

##### Ours with their blending: not smooth.
my_Djuren_surface.compute_blend_weights(flag='djuren') #blend type

my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='equal') #vertex func type, onering coords type
mesh_o3d_angles, mesh_o3d_plain = my_Djuren_surface.build_output_surface()

o3d.visualization.draw_geometries([mesh_o3d_angles])
o3d.visualization.draw_geometries([mesh_o3d_plain])
'''

'''
##### Theirs with their blending
my_Djuren_surface.compute_blend_weights(flag='djuren') #blend type

my_Djuren_surface.forward(function_flag='constant', onering_coords_flag='djuren') #vertex func type, onering coords type
mesh_o3d_angles, mesh_o3d_plain = my_Djuren_surface.build_output_surface()

o3d.visualization.draw_geometries([mesh_o3d_angles])
o3d.visualization.draw_geometries([mesh_o3d_plain])
'''


In [ ]:

print([ my_Djuren_surface.bary_weight(torch.tensor([0.0,0.0]).unsqueeze(0), 0),
my_Djuren_surface.bary_weight(torch.tensor([0.0,0.0]).unsqueeze(0), 1),
my_Djuren_surface.bary_weight(torch.tensor([0.0,0.0]).unsqueeze(0), 2) ])

print([ my_Djuren_surface.bary_weight(torch.tensor([1.0,0.0]).unsqueeze(0), 0),
my_Djuren_surface.bary_weight(torch.tensor([1.0,0.0]).unsqueeze(0), 1),
my_Djuren_surface.bary_weight(torch.tensor([1.0,0.0]).unsqueeze(0), 2) ])


print([ my_Djuren_surface.bary_weight(torch.tensor([0.5,np.sqrt(3)/2]).unsqueeze(0), 0),
my_Djuren_surface.bary_weight(torch.tensor([0.5,np.sqrt(3)/2]).unsqueeze(0), 1),
my_Djuren_surface.bary_weight(torch.tensor([0.5,np.sqrt(3)/2]).unsqueeze(0), 2) ])